# SqueakView Run Analysis Demo

This notebook visualizes one aligned run using the RP2040 `CAMERA_HIGH` timestamps as the common time base. It expects to live inside a run `analysis/` directory beside `aligned_all.csv`, `aligned_frames.csv`, `aligned_events.csv`, `aligned_detections.csv`, and `alignment_summary.json`.


In [ ]:
from pathlib import Path
import json
import math
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Resolve paths. If this notebook is moved out of analysis/, fall back to runs/.latest_run.
HERE = Path.cwd()
if (HERE / "aligned_all.csv").exists():
    ANALYSIS_DIR = HERE
    RUN_DIR = ANALYSIS_DIR.parent
else:
    REPO_ROOT = next(
        (
            parent
            for parent in [HERE, *HERE.parents]
            if (parent / "pyproject.toml").exists() and (parent / "squeakview").exists()
        ),
        HERE,
    )
    RUN_DIR = Path((REPO_ROOT / "runs" / ".latest_run").read_text().strip())
    ANALYSIS_DIR = RUN_DIR / "analysis"

print(f"RUN_DIR:      {RUN_DIR}")
print(f"ANALYSIS_DIR: {ANALYSIS_DIR}")


## Load Aligned Tables

`aligned_all.csv` is the single long-format table. The split tables are easier for plotting specific views:

- `aligned_frames.csv`: one row per recorded raw video frame
- `aligned_events.csv`: one row per serial/controller event
- `aligned_detections.csv`: one row per inference detection
- `alignment_summary.json`: run-level validation and counts


In [ ]:
summary = json.loads((ANALYSIS_DIR / "alignment_summary.json").read_text())
all_df = pd.read_csv(ANALYSIS_DIR / "aligned_all.csv")
frames = pd.read_csv(ANALYSIS_DIR / "aligned_frames.csv")
events = pd.read_csv(ANALYSIS_DIR / "aligned_events.csv")
detections = pd.read_csv(ANALYSIS_DIR / "aligned_detections.csv")

numeric_cols = {
    "frames": [
        "camera_frame_id", "ttl_count", "frame_rp2040_us", "frame_time_s", "frame_pts_ns",
        "frame_pts_s", "duration_ns", "has_ttl", "has_detection", "detection_count",
    ],
    "events": [
        "serial_index", "rp2040_time_us", "event_time_s", "count", "previous_frame_id",
        "offset_from_previous_frame_ms", "nearest_frame_id", "offset_from_nearest_frame_ms",
    ],
    "detections": [
        "detection_index", "camera_frame_id", "ttl_count", "detection_rp2040_us", "detection_time_s",
        "frame_pts_ns", "frame_pts_s", "raw_frame_mapping_ok", "raw_frame_mapping_pts_ns",
        "raw_frame_mapping_delta", "conf", "x", "y", "w", "h", "original_frame", "original_ts_us",
    ],
}

for col in numeric_cols["frames"]:
    if col in frames:
        frames[col] = pd.to_numeric(frames[col], errors="coerce")
for col in numeric_cols["events"]:
    if col in events:
        events[col] = pd.to_numeric(events[col], errors="coerce")
for col in numeric_cols["detections"]:
    if col in detections:
        detections[col] = pd.to_numeric(detections[col], errors="coerce")

print(f"frames:     {len(frames):,}")
print(f"events:     {len(events):,}")
print(f"detections: {len(detections):,}")
print(f"all rows:   {len(all_df):,}")

display(frames.head(3))
display(events.head(3))
display(detections.head(3))


## Alignment Health Check

This is the first block to inspect after every run. For a clean run, the mismatch/fallback/drop counts should be zero and the video frame count should match `frames.csv`.


In [ ]:
counts = pd.Series(summary.get("counts", {}), name="value").to_frame()
validation = pd.Series(summary.get("validation", {}), name="value").to_frame()
markers = pd.Series(summary.get("markers", {}), name="ttl_count").to_frame()

print("Counts")
display(counts)
print("Validation")
display(validation)
print("Markers")
display(markers)

required_zero = [
    "frame_gaps_detected",
    "frames_missing_ttl",
    "drop_events",
]
for key in required_zero:
    value = summary.get("counts", {}).get(key)
    if value not in (0, None):
        print(f"WARNING: {key} = {value}")

for key in [
    "detection_mapping_failed_rows",
    "detection_mapping_fallback_rows",
    "detections_missing_frame_count",
    "detection_ts_mismatch_count",
    "detection_pts_mismatch_count",
]:
    value = summary.get("validation", {}).get(key)
    if value not in (0, None):
        print(f"WARNING: {key} = {value}")


## Frame Timing

This compares the microcontroller TTL interval against the video/GStreamer PTS interval. The TTL interval is the acquisition clock we care about most.


In [ ]:
frames = frames.sort_values("camera_frame_id").reset_index(drop=True)
frames["ttl_interval_ms"] = frames["frame_rp2040_us"].diff() / 1_000.0
frames["pts_interval_ms"] = frames["frame_pts_ns"].diff() / 1_000_000.0
expected_ms = 1_000.0 / 30.0

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
axes[0].plot(frames["camera_frame_id"], frames["ttl_interval_ms"], ".-", markersize=3, linewidth=0.8, label="RP2040 CAMERA_HIGH")
axes[0].plot(frames["camera_frame_id"], frames["pts_interval_ms"], ".-", markersize=3, linewidth=0.8, alpha=0.8, label="GStreamer PTS")
axes[0].axhline(expected_ms, color="black", linestyle="--", linewidth=1, label="30 fps target")
axes[0].set_title("Frame-to-frame interval")
axes[0].set_xlabel("camera_frame_id")
axes[0].set_ylabel("interval (ms)")
axes[0].legend(loc="upper right")

sns.histplot(frames["ttl_interval_ms"].dropna(), bins=40, ax=axes[1], color="tab:blue", label="RP2040")
sns.histplot(frames["pts_interval_ms"].dropna(), bins=40, ax=axes[1], color="tab:orange", label="PTS", alpha=0.55)
axes[1].axvline(expected_ms, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Interval distribution")
axes[1].set_xlabel("interval (ms)")
axes[1].legend()
plt.tight_layout()

interval_summary = frames[["ttl_interval_ms", "pts_interval_ms"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(interval_summary)


## Detection Coverage Over Time

This shows which frames have detections and overlays non-camera serial events on the same RP2040-derived seconds axis.


In [ ]:
non_camera_events = events[~events["eventType"].isin(["CAMERA_HIGH", "CAMERA_LOW"])].copy()
non_camera_events = non_camera_events.dropna(subset=["event_time_s"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(frames["frame_time_s"], frames["detection_count"], color="tab:blue", linewidth=1.5, label="detections per frame")
ax.scatter(detections["detection_time_s"], detections["conf"], s=12, color="tab:green", alpha=0.55, label="detection confidence")

label_y = max(1.0, float(np.nanmax(frames["detection_count"])) if len(frames) else 1.0)
for _, row in non_camera_events.iterrows():
    event_name = row.get("event_name") or row.get("eventType")
    t = row["event_time_s"]
    ax.axvline(t, color="tab:red", alpha=0.18, linewidth=0.8)
    if row["eventType"] in {"POKE_START", "POKE_END", "REWARD", "MARKER", "ACK_START", "ACK_STOP", "STRIP_ON", "STRIP_OFF"}:
        ax.text(t, label_y + 0.05, str(event_name), rotation=90, va="bottom", ha="center", fontsize=8, alpha=0.75)

ax.set_title("Detections and serial events on RP2040 time base")
ax.set_xlabel("time from first recorded CAMERA_HIGH (s)")
ax.set_ylabel("detection count / confidence")
ax.legend(loc="upper right")
plt.tight_layout()


## Serial Event Raster

This makes it easier to see task/sensor events without the dense camera pulse stream.


In [ ]:
plot_events = events[~events["eventType"].isin(["CAMERA_HIGH", "CAMERA_LOW"])].copy()
plot_events = plot_events.dropna(subset=["event_time_s"])
plot_events["event_label"] = plot_events["eventType"].fillna("UNKNOWN")

if plot_events.empty:
    print("No non-camera serial events found.")
else:
    order = sorted(plot_events["event_label"].unique())
    fig, ax = plt.subplots(figsize=(14, max(3, 0.35 * len(order) + 2)))
    sns.scatterplot(
        data=plot_events,
        x="event_time_s",
        y="event_label",
        hue="side" if "side" in plot_events.columns else None,
        style="context" if "context" in plot_events.columns else None,
        s=70,
        ax=ax,
    )
    ax.set_title("Non-camera serial events")
    ax.set_xlabel("time from first recorded CAMERA_HIGH (s)")
    ax.set_ylabel("event type")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0)
    plt.tight_layout()

    display(plot_events["eventType"].value_counts().to_frame("count"))


## Detection-to-Frame Mapping Proof

For new runs, every detection should usually be `pts_match`. Any fallback rows should be inspected before using the detections for analysis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

method_counts = detections["raw_frame_mapping_method"].fillna("missing").value_counts()
method_counts.plot(kind="bar", ax=axes[0], color="tab:purple")
axes[0].set_title("Detection mapping methods")
axes[0].set_xlabel("method")
axes[0].set_ylabel("rows")
axes[0].tick_params(axis="x", rotation=30)

map_check = detections.merge(
    frames[["camera_frame_id", "frame_pts_ns", "frame_time_s"]],
    on="camera_frame_id",
    how="left",
    suffixes=("_det", "_frame"),
)
map_check["ts_delta_us"] = map_check["original_ts_us"] - (map_check["frame_pts_ns_frame"] // 1_000)
map_check["mapping_pts_delta_ns"] = map_check["raw_frame_mapping_pts_ns"] - map_check["frame_pts_ns_frame"]

sns.histplot(map_check["ts_delta_us"].dropna(), bins=40, ax=axes[1], color="tab:green")
axes[1].set_title("Detection timestamp minus frame PTS")
axes[1].set_xlabel("delta (us)")
axes[1].set_ylabel("detections")
plt.tight_layout()

display(map_check[[
    "detection_index", "camera_frame_id", "raw_frame_mapping_method", "raw_frame_mapping_ok",
    "original_ts_us", "frame_pts_ns_frame", "ts_delta_us", "mapping_pts_delta_ns",
]].head(20))

problem_rows = map_check[
    (map_check["raw_frame_mapping_ok"].fillna(1) != 1)
    | (map_check["camera_frame_id"].isna())
    | (map_check["ts_delta_us"].abs() > 1)
]
print(f"problem mapping rows: {len(problem_rows):,}")
if len(problem_rows):
    display(problem_rows.head(20))


## Bounding Box Trajectory

This plots the detected object center over time and in image coordinates. For multiple detections per frame, the highest-confidence detection is used.


In [ ]:
if detections.empty:
    print("No detections to plot.")
else:
    det = detections.copy()
    det["cx"] = det["x"] + det["w"] / 2.0
    det["cy"] = det["y"] + det["h"] / 2.0
    best = det.sort_values(["camera_frame_id", "conf"], ascending=[True, False]).drop_duplicates("camera_frame_id")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(best["detection_time_s"], best["cx"], label="bbox center x", linewidth=1.2)
    axes[0].plot(best["detection_time_s"], best["cy"], label="bbox center y", linewidth=1.2)
    axes[0].set_title("Best detection center over time")
    axes[0].set_xlabel("time (s)")
    axes[0].set_ylabel("pixel")
    axes[0].legend()

    sc = axes[1].scatter(best["cx"], best["cy"], c=best["detection_time_s"], s=18, cmap="viridis")
    axes[1].invert_yaxis()
    axes[1].set_aspect("equal", adjustable="box")
    axes[1].set_title("Best detection path in image coordinates")
    axes[1].set_xlabel("x pixel")
    axes[1].set_ylabel("y pixel")
    plt.colorbar(sc, ax=axes[1], label="time (s)")
    plt.tight_layout()


## Pose Keypoint Trajectory

This parses `kpt_names_json` / `kpt_values_json` into a long table. Change `KEYPOINTS_TO_PLOT` to focus on different body/task landmarks.


In [ ]:
def build_pose_long(detections_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in detections_df.iterrows():
        names_raw = row.get("kpt_names_json")
        values_raw = row.get("kpt_values_json")
        if pd.isna(names_raw) or pd.isna(values_raw):
            continue
        try:
            names = json.loads(names_raw)
            values = json.loads(values_raw)
        except Exception:
            continue
        for name, triplet in zip(names, values):
            if len(triplet) < 3:
                continue
            rows.append({
                "detection_index": row.get("detection_index"),
                "camera_frame_id": row.get("camera_frame_id"),
                "time_s": row.get("detection_time_s"),
                "keypoint": name,
                "x": float(triplet[0]),
                "y": float(triplet[1]),
                "score": float(triplet[2]),
                "det_conf": row.get("conf"),
            })
    return pd.DataFrame(rows)

pose = build_pose_long(detections)
print(f"pose rows: {len(pose):,}")
if not pose.empty:
    display(pose.head())
    display(pose.groupby("keypoint")["score"].describe().sort_values("mean", ascending=False))


In [ ]:
KEYPOINTS_TO_PLOT = ["nose", "head", "back", "tail_base"]
MIN_KEYPOINT_SCORE = 0.5

if pose.empty:
    print("No pose rows to plot.")
else:
    p = pose[(pose["keypoint"].isin(KEYPOINTS_TO_PLOT)) & (pose["score"] >= MIN_KEYPOINT_SCORE)].copy()
    if p.empty:
        print("No keypoints pass the selected score threshold.")
    else:
        fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
        sns.lineplot(data=p, x="time_s", y="x", hue="keypoint", estimator=None, units="keypoint", ax=axes[0])
        axes[0].set_title("Keypoint x over time")
        axes[0].set_ylabel("x pixel")
        sns.lineplot(data=p, x="time_s", y="y", hue="keypoint", estimator=None, units="keypoint", ax=axes[1], legend=False)
        axes[1].set_title("Keypoint y over time")
        axes[1].set_xlabel("time (s)")
        axes[1].set_ylabel("y pixel")
        plt.tight_layout()

        fig, ax = plt.subplots(figsize=(7, 7))
        sns.scatterplot(data=p, x="x", y="y", hue="keypoint", size="score", sizes=(15, 90), alpha=0.75, ax=ax)
        ax.invert_yaxis()
        ax.set_aspect("equal", adjustable="box")
        ax.set_title("Keypoint positions in image coordinates")
        plt.tight_layout()


## Event-Aligned Windows

Use this to inspect behavior around a specific serial event type. It returns detection/frame rows in a peri-event window.


In [ ]:
def event_locked_detections(event_type: str, pre_s: float = 1.0, post_s: float = 2.0) -> pd.DataFrame:
    anchors = events[(events["eventType"] == event_type) & events["event_time_s"].notna()].copy()
    chunks = []
    for idx, ev in anchors.iterrows():
        t0 = float(ev["event_time_s"])
        window = detections[(detections["detection_time_s"] >= t0 - pre_s) & (detections["detection_time_s"] <= t0 + post_s)].copy()
        window["anchor_index"] = idx
        window["anchor_event_type"] = event_type
        window["time_from_event_s"] = window["detection_time_s"] - t0
        chunks.append(window)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

EVENT_TYPE = "POKE_START"
locked = event_locked_detections(EVENT_TYPE, pre_s=1.0, post_s=2.0)
print(f"{EVENT_TYPE} locked detections: {len(locked):,}")
if not locked.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.scatterplot(data=locked, x="time_from_event_s", y="conf", hue="anchor_index", palette="tab20", legend=False, ax=ax)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Detections around {EVENT_TYPE}")
    ax.set_xlabel("time from event (s)")
    ax.set_ylabel("detection confidence")
    plt.tight_layout()
    display(locked.head(20))
else:
    print(f"No {EVENT_TYPE} events or no detections in the selected window.")


## Optional Raw Video Preview

Set `RUN_PREVIEW = True` to extract one PNG frame from the raw MP4 using `ffmpeg`. This is useful for confirming that a detection row visually corresponds to the expected raw frame.


In [ ]:
RUN_PREVIEW = True
DETECTION_INDEX_TO_PREVIEW = 610
KEYPOINT_SCORE_THRESHOLD = 0.35

from matplotlib.patches import Rectangle

CLASS_COLORS = {
    "mouse": "lime",
    "landmarks": "magenta",
}
DEFAULT_DETECTION_COLOR = "cyan"


def _selected_detection_by_index(detections_df: pd.DataFrame, detection_index: int) -> pd.Series:
    match = detections_df[detections_df["detection_index"] == detection_index]
    if not match.empty:
        return match.iloc[0]
    # Fallback keeps old behavior if DETECTION_INDEX_TO_PREVIEW is used as a row position.
    return detections_df.iloc[detection_index]


def _parse_keypoints(det_row: pd.Series, min_score: float) -> list[tuple[str, float, float, float]]:
    if pd.isna(det_row.get("kpt_names_json")) or pd.isna(det_row.get("kpt_values_json")):
        return []
    try:
        names = json.loads(det_row["kpt_names_json"])
        values = json.loads(det_row["kpt_values_json"])
    except Exception as exc:
        print(f"Could not parse keypoints for detection {det_row.get('detection_index')}: {exc}")
        return []

    parsed = []
    for name, triplet in zip(names, values):
        if len(triplet) < 3:
            continue
        kx, ky, score = map(float, triplet[:3])
        if score >= min_score:
            parsed.append((name, kx, ky, score))
    return parsed


if RUN_PREVIEW and not detections.empty:
    selected = _selected_detection_by_index(detections, DETECTION_INDEX_TO_PREVIEW)
    frame_id = int(selected["camera_frame_id"])
    frame_detections = detections[detections["camera_frame_id"] == frame_id].copy()
    frame_detections = frame_detections.sort_values(["class_label", "conf"], ascending=[True, False])

    frame_match = frames[frames["camera_frame_id"] == frame_id]
    if frame_match.empty:
        raise ValueError(f"No frame row found for camera_frame_id={frame_id}")
    frame_row = frame_match.iloc[0]

    video_path = Path(frame_row["record_segment_file"])
    t_s = float(frame_row["frame_pts_s"])
    out_png = ANALYSIS_DIR / f"preview_frame_{frame_id:06d}.png"

    # Extract the raw video frame nearest this frame PTS.
    cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-ss", f"{t_s:.6f}", "-i", str(video_path),
        "-frames:v", "1", str(out_png),
    ]
    subprocess.run(cmd, check=True)

    img = plt.imread(out_png)
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(img, cmap="gray")

    for _, det_row in frame_detections.iterrows():
        label = str(det_row.get("class_label", "detection"))
        color = CLASS_COLORS.get(label, DEFAULT_DETECTION_COLOR)
        x, y, w, h = [float(det_row[v]) for v in ["x", "y", "w", "h"]]

        rect = Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5)
        ax.add_patch(rect)
        ax.text(
            x,
            max(0, y - 8),
            f"{int(det_row['detection_index'])}: {label} conf={float(det_row['conf']):.3f}",
            color="black",
            fontsize=9,
            bbox={"facecolor": color, "edgecolor": "none", "alpha": 0.85, "pad": 2},
        )

        keypoint_rows = _parse_keypoints(det_row, KEYPOINT_SCORE_THRESHOLD)
        if keypoint_rows:
            kx = [r[1] for r in keypoint_rows]
            ky = [r[2] for r in keypoint_rows]
            scores = [r[3] for r in keypoint_rows]
            ax.scatter(
                kx,
                ky,
                s=[30 + 90 * s for s in scores],
                c=color,
                edgecolors="black",
                linewidths=0.8,
                alpha=0.9,
            )
            for name, px, py, score in keypoint_rows:
                ax.text(
                    px + 4,
                    py + 4,
                    name,
                    color="white",
                    fontsize=7,
                    bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55, "pad": 1},
                )

    ax.set_title(
        f"Frame {frame_id} | TTL {int(frame_row['ttl_count'])} | t={t_s:.3f}s | "
        f"detections={len(frame_detections)}"
    )
    ax.set_xlim(0, img.shape[1])
    ax.set_ylim(img.shape[0], 0)
    ax.axis("off")
    plt.tight_layout()

    display(frame_detections[[
        "detection_index", "camera_frame_id", "ttl_count", "class_label", "conf",
        "raw_frame_mapping_method", "raw_frame_mapping_ok", "x", "y", "w", "h",
    ]])
else:
    print("Preview disabled. Set RUN_PREVIEW = True to extract and overlay a frame with ffmpeg.")
